<a href="https://colab.research.google.com/github/rajilsaj/nasa-mosaics-project/blob/xgboost/notebooks/04_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

drive.mount('/content/drive')

SPLIT_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project/data/splits"
WINDOW_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows"
OUTPUT_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows"

df = pd.read_csv(f"{WINDOW_DIR}/train_balanced.csv")
print("Rows:", len(df))
print("Windows:", df["window_id"].nunique())

Mounted at /content/drive
Rows: 27000
Windows: 450


In [3]:
def compute_features(window_df):
    p = window_df["PRESSURE"].values

    features = {}

    # Trend
    x = np.arange(len(p))
    slope = np.polyfit(x, p, 1)[0]
    features["slope"] = slope

    # Pressure drop
    features["pressure_drop"] = p[0] - p.min()
    features["min_position"] = np.argmin(p) / len(p)

    # Statistics
    features["mean"] = p.mean()
    features["std"] = p.std()
    features["range"] = p.max() - p.min()

    # Label
    features["label"] = window_df["label"].iloc[0]

    return features


In [4]:
feature_rows = []

for window_id, window_data in df.groupby("window_id"):
    features = compute_features(window_data)
    feature_rows.append(features)

features_df = pd.DataFrame(feature_rows)


In [5]:
features_df.to_csv(f"{WINDOW_DIR}/train_features.csv", index=False)
